# Problem Set 1: figures and tables (Questions 6-11)
Builds every figure and table of the write-up from the **already estimated** full models in `estimates_full/` (no re-estimation), writes them to `output/figures`, `output/tables`, and writes the Overleaf skeleton `output/overleaf/main.tex`.

What is *computed* here (not re-estimated): the true-parameter simulations (as in `pset_io.ipynb`), per-start betas / standard errors / contraction errors at the stored per-start sigma minimizers (8c, 8e, 11a), the cost recovery under three conduct models (9d), the merger counterfactual (10) and the diagnostics of Q11.

## 0. Setup

In [ ]:
using MAT, DataFrames, Statistics, LinearAlgebra, JLD2, Distributions, KernelDensity, Plots, Optim, Printf, Random
include("./blp_optimize/blp_new.jl")
using .blp
mkpath("output/figures"); mkpath("output/tables"); mkpath("output/overleaf")

### Style, registry and writers

In [ ]:
# ---- palette: blues and reds. J=3 blue / J=5 red; real blue / simulated red ----
const BLUE, LBLUE, DBLUE = "#2166ac", "#7fb2d9", "#0b2e59"
const RED, LRED, DRED    = "#b2182b", "#e8837c", "#6e0f1a"
const GRAYD = "#555555"

default(framestyle = :axes, grid = true, gridalpha = 0.25, gridlinewidth = 0.5, gridstyle = :solid,
        tickfontsize = 10, guidefontsize = 11, legendfontsize = 10, fontfamily = "Helvetica",
        dpi = 300, size = (640, 420), legend = :best, foreground_color_legend = nothing,
        background_color_legend = RGBA(1, 1, 1, 0.7), margin = 4Plots.mm)

# ---- registry: every figure / table that is written, used to build main.tex ----
ITEMS = NamedTuple[]

function save_fig(p, name; q, caption, source)
    savefig(p, "output/figures/$(name).png")
    savefig(p, "output/figures/$(name).pdf")
    push!(ITEMS, (q = q, kind = :fig, name = name, caption = caption, source = source))
    return p
end

function save_tab(lines, name; q, caption, source)
    open("output/tables/$(name).tex", "w") do io
        foreach(l -> println(io, l), lines)
    end
    push!(ITEMS, (q = q, kind = :tab, name = name, caption = caption, source = source))
    return nothing
end

# ---- number formatting ----
fmt(x; d = 3) = (ismissing(x) || !isfinite(x)) ? "--" : replace(Printf.format(Printf.Format("%." * string(d) * "f"), x), "-" => "\$-\$")
function fsci(x; d = 2)
    (ismissing(x) || !isfinite(x)) && return "--"
    x == 0 && return "0"
    e = floor(Int, log10(abs(x))); m = x / 10.0^e
    return "\$" * replace(Printf.format(Printf.Format("%." * string(d) * "f"), m), "-" => "-") * "\\times10^{" * string(e) * "}\$"
end
stars(t) = (p = 2 * (1 - cdf(Normal(), abs(t))); p < 0.01 ? "***" : p < 0.05 ? "**" : p < 0.1 ? "*" : "")
star_tex(t) = (s = stars(t); isempty(s) ? "" : "\$^{" * s * "}\$")
est_cell(b, se; d = 3) = !isfinite(b) ? "--" : fmt(b; d = d) * (isfinite(se) && se > 0 ? star_tex(b / se) : "")
se_cell(se; d = 3) = !isfinite(se) ? "(n.a.)" : se > 1e4 ? "(" * fsci(se) * ")" : "(" * fmt(se; d = d) * ")"
const NOTE_NA = " (n.a.): estimated variance not positive; very large standard errors indicate weak identification."
bold(s) = "\\textbf{" * s * "}"

# ---- LaTeX tabular with booktabs; rules = row indices after which a \midrule is inserted ----
function tabular(header, rows; align = nothing, rules = Int[])
    nc = length(header)
    align = something(align, "l" * "r"^(nc - 1))
    L = ["\\begin{tabular}{$align}", "\\toprule", join(header, " & ") * " \\\\", "\\midrule"]
    for (i, r) in enumerate(rows)
        push!(L, join(r, " & ") * " \\\\")
        i in rules && push!(L, "\\midrule")
    end
    push!(L, "\\bottomrule", "\\end{tabular}")
    return L
end

# ---- descriptives: mean, sd, median, q25, q75 ----
desc(x) = (v = filter(isfinite, vec(x)); [mean(v), std(v), median(v), quantile(v, 0.25), quantile(v, 0.75)])
const DESC_HEADER = ["Mean", "SD", "Median", "Q25", "Q75"]
desc_cells(x; d = 3) = [fmt(v; d = d) for v in desc(x)]

### Plot helpers (histogram + kernel density, 45-degree scatter)

In [ ]:
# Plot text without LaTeX (GR would call `latex` for $...$): strip the TeX used in the LaTeX captions
function ptxt(s)
    s = replace(s, "\\Delta" => "\u0394", "\\delta" => "\u03b4", "\\mu" => "\u03bc", "\\sigma" => "\u03c3", "\\eta" => "\u03b7", "\\alpha" => "\u03b1", "\\cdot" => "\u00b7", "\\mathrm" => "")
    s = replace(s, "\$" => "", "{" => "", "}" => "", "\\" => "")
    return s
end

# series = [(data, label, color)]. Histograms on common bins, normalised to a density, with an overlaid kernel density.
function hist_kde(series; xlabel, ylabel = "Density", nbins = 30, clipq = 0.0, legend = :topright)
    pooled = vcat([vec(s[1]) for s in series]...)
    pooled = filter(isfinite, pooled)
    lo, hi = clipq > 0 ? (quantile(pooled, clipq), quantile(pooled, 1 - clipq)) : extrema(pooled)
    edges = range(lo, hi, length = nbins + 1)
    p = plot(; xlabel = ptxt(xlabel), ylabel = ptxt(ylabel), legend)
    for (x, l, c) in series
        v = filter(z -> isfinite(z) && lo <= z <= hi, vec(x))
        histogram!(p, v; bins = edges, normalize = :pdf, color = c, alpha = 0.35, linecolor = :white, linewidth = 0.4, label = ptxt(l))
    end
    for (x, l, c) in series
        v = filter(z -> isfinite(z) && lo <= z <= hi, vec(x))
        k = kde(v)
        keep_ = (k.x .>= lo) .& (k.x .<= hi)
        plot!(p, k.x[keep_], k.density[keep_]; color = c, lw = 2.2, label = false)
    end
    xlims!(p, lo, hi)
    return p
end

# series = [(x, y, label, color)]; points plus a dashed 45-degree line
function scat45(series; xlabel, ylabel, line_label = "45° line", ms = 3.5, alpha = 0.6, legend = :topleft, qlim = 0.0)
    allv = vcat([vcat(vec(s[1]), vec(s[2])) for s in series]...)
    allv = filter(isfinite, allv)
    lo, hi = qlim > 0 ? (quantile(allv, qlim), quantile(allv, 1 - qlim)) : extrema(allv); pad = 0.03 * (hi - lo); lo -= pad; hi += pad
    p = plot([lo, hi], [lo, hi]; color = GRAYD, ls = :dash, lw = 1.5, label = ptxt(line_label), xlabel = ptxt(xlabel), ylabel = ptxt(ylabel), legend, xlims = (lo, hi), ylims = (lo, hi))
    for (x, y, l, c) in series
        scatter!(p, vec(x), vec(y); label = ptxt(l), color = c, ms = ms, alpha = alpha, markerstrokewidth = 0)
    end
    return p
end

### Truth and stored objects

In [ ]:
beta = [5, 1, 1]; alpha_bar = -1; gamma = [2, 1, 1]; sigma = 1.0

d_T10_J3  = matread("/Users/erickom/Documents/2026 Fall/nu-IO/psets/Simulation Data/10markets3products.mat")
d_T100_J3 = matread("/Users/erickom/Documents/2026 Fall/nu-IO/psets/Simulation Data/100markets3products.mat")
d_T100_J5 = matread("/Users/erickom/Documents/2026 Fall/nu-IO/psets/Simulation Data/100markets5products.mat")

# True-parameter model: data + true xi, true costs, simulated with the true parameters (as in pset_io cells 3-9)
function process_truth(d, errors)
    J, T = size(d["P_opt"])
    p = reshape(d["P_opt"], (T * J, 1)); shares = reshape(d["shares"], (T * J, 1))
    m = blp.BLPDemand(BLPData = blp.BLPData(T, J, d["x1"], p, shares, errors))
    m.xi_real = reshape(d["xi_all"], (T * J, 1))
    m.cost = gamma[1] .+ (gamma[2] * d["w"] + gamma[3] * d["Z"] + d["eta"])
    m.markup = (m.p - m.cost) ./ m.p
    m.profit = (m.p - m.cost) .* m.shares
    m.beta .= vcat(beta, [alpha_bar]); m.sigma = sigma
    blp.simulate_BLP!(m, delta = m.X_all * m.beta + m.xi_real, sigma = sigma, all = true)
    return m
end

errors_T100 = log.(d_T100_J3["alphas"] .- 1)       # consumer draws, common to J=3 and J=5
truth_J3 = process_truth(d_T100_J3, errors_T100)
truth_J5 = process_truth(d_T100_J5, errors_T100)

dem(m) = m isa blp.BLP ? m.BLPDemand : m
ld(path) = load(path)["model"]
ldf(path) = load(path)["failures"]

Stored estimates (all from `estimates_full/`). Explicit paths, one line per model.

In [ ]:
m8c_only    = ld("estimates_full/q8c_blp_only_T100_J3.jdl2")
m8c_haus    = ld("estimates_full/q8c_blp_hausman_T100_J3.jdl2")
m8e_only    = ld("estimates_full/q8e_blp_only_T10_J3.jdl2")
m8e_haus    = ld("estimates_full/q8e_blp_hausman_T10_J3.jdl2")
m8f_price   = ld("estimates_full/q8f_blp_price_T100_J3.jdl2")
m8f_price_h = ld("estimates_full/q8f_blp_price_hausman_T100_J3.jdl2")
m9a_w       = ld("estimates_full/q9a_blp_w_T100_J3.jdl2")
m9a_haus_w  = ld("estimates_full/q9a_blp_hausman_w_T100_J3.jdl2")
m9b_w       = ld("estimates_full/q9b_blp_w_T10_J3.jdl2")
m9b_haus_w  = ld("estimates_full/q9b_blp_hausman_w_T10_J3.jdl2")
m9e_bert    = ld("estimates_full/q9e_system_bertrand_T100_J3.jdl2")
m9e_comp    = ld("estimates_full/q9e_system_competition_T100_J3.jdl2")
m9e_coll    = ld("estimates_full/q9e_system_collusion_T100_J3.jdl2")
m11_blp     = ld("estimates_full/q11a_blp_only_T100_J5.jdl2")
m11_gh      = ld("estimates_full/q11a_gh_only_T100_J5.jdl2")
m11_ghh     = ld("estimates_full/q11a_gh_hausman_T100_J5.jdl2")

f8c_only = ldf("estimates_full/q8c_blp_only_T100_J3_failed_starts.jdl2")
f8e_only = ldf("estimates_full/q8e_blp_only_T10_J3_failed_starts.jdl2")
f11_blp  = ldf("estimates_full/q11a_blp_only_T100_J5_failed_starts.jdl2")
f11_gh   = ldf("estimates_full/q11a_gh_only_T100_J5_failed_starts.jdl2")
f11_ghh  = ldf("estimates_full/q11a_gh_hausman_T100_J5_failed_starts.jdl2")

### Estimates-table helpers

In [ ]:
const ROWLAB = ["Constant", "\$x_2\$ (uniform)", "\$x_3\$ (normal)", "Price coefficient", "\$\\gamma_0\$ (cost constant)", "\$\\gamma_z\$ (\$z_{jt}\$)", "\$\\gamma_w\$ (\$w_j\$)", "\$\\sigma_\\alpha\$"]
const TRUTHV = [5.0, 1.0, 1.0, -1.0, 2.0, 1.0, 1.0, 1.0]

safe_se(V) = [v > 0 ? sqrt(v) : NaN for v in diag(V)]

# Layout: [const, x2, x3, price, gamma0, gamma_z, gamma_w, sigma]; supply entries are NaN for demand-only models
function par(m)
    pr = vec(m.params); se = safe_se(m.Var)
    if m isa blp.BLP
        return (est = copy(pr), se = se)
    else
        return (est = vcat(pr[1:4], fill(NaN, 3), pr[5]), se = vcat(se[1:4], fill(NaN, 3), se[5]))
    end
end
obj(m) = minimum(Optim.minimum.(m.opt_res))

# Regression-style table: estimates with stars, SE in parentheses, true values, objective and N
function reg_table(colnames, models; truth = true)
    P = [par(m) for m in models]
    sys = any(m isa blp.BLP for m in models)
    ri = sys ? (1:8) : [1, 2, 3, 4, 8]
    header = vcat([""], truth ? ["True"] : String[], colnames)
    rows = Vector{Vector{String}}()
    for i in ri
        push!(rows, vcat([ROWLAB[i]], truth ? [fmt(TRUTHV[i])] : String[], [est_cell(p.est[i], p.se[i]) for p in P]))
        push!(rows, vcat([""], truth ? [""] : String[], [se_cell(p.se[i]) for p in P]))
    end
    nr = length(rows)
    push!(rows, vcat(["GMM objective"], truth ? [""] : String[], [fsci(obj(m)) for m in models]))
    push!(rows, vcat(["Observations (\$JT\$)"], truth ? [""] : String[], [string(dem(m).J * dem(m).T) for m in models]))
    return tabular(header, rows; rules = [nr])
end

# Estimates at a given sigma (one start): beta by the closed-form linear step, SE from the GMM sandwich (as in estimate_blp!),
# and the contraction error  max |log s_obs - log s(delta; sigma)|.
function at_sigma(m, sg; toler = 1e-10)
    mm = deepcopy(m)
    mm.M = blp.gmm_M(mm.X_all, mm.Z, mm.W)
    delta = blp.contraction(mm, sg, toler = toler, max_iter = 10000)
    v = blp.random_coef(mm, sg)
    s_sim = zeros(length(delta)); buf = zeros(mm.J)
    blp.share_sim!(s_sim, buf, mm, vec(delta), v)
    err = maximum(abs.(log.(vec(mm.shares)) .- log.(s_sim)))
    mm.sigma = sg; mm.delta = delta
    mm.beta = blp.gmm_beta(mm.X_all, mm.Z, mm.W, delta)
    mm.xi = mm.M * delta
    blp.simulate_BLP!(mm, delta = delta, sigma = sg, all = true)
    N = mm.J * mm.T
    omega_var = mm.Z' * ((mm.xi .^ 2) .* mm.Z) ./ N
    Gamma = 1 / N * mm.Z' * hcat(-mm.X_all, mm.grad)
    GWG = Gamma' * mm.W * Gamma
    Var = (GWG \ (Gamma' * mm.W * omega_var * mm.W * Gamma)) / GWG / N
    return (est = vcat(vec(mm.beta), sg), se = safe_se(Var), err = err)
end

# One row per start (estimates by starting point); the chosen start (lowest objective) is in bold
function by_start_table(m)
    R = m.opt_res
    objs = Optim.minimum.(R); best = argmin(objs)
    rows = Vector{Vector{String}}()
    for (i, r) in enumerate(R)
        a = at_sigma(m, Optim.minimizer(r)[1])
        cells = [est_cell(a.est[k], a.se[k]) * " " * se_cell(a.se[k]) for k in 1:5]
        row = vcat([fmt(Optim.initial_x(r)[1])], cells, [fsci(objs[i])])
        push!(rows, i == best ? bold.(row) : row)
    end
    d = dem(m)
    push!(rows, vcat(["True"], [fmt(v) for v in TRUTHV[[1, 2, 3, 4, 8]]], [""]))
    header = ["Start \$\\sigma_0\$", "Constant", "\$x_2\$", "\$x_3\$", "Price", "\$\\sigma_\\alpha\$", "GMM obj."]
    return tabular(header, rows; align = "rccccc" * "c", rules = [length(R)])
end

# Convergence criterion and contraction error by start (failed starts listed with their initial sigma)
function convergence_table(m, failures)
    R = m.opt_res
    rows = Vector{Vector{String}}()
    objs = Optim.minimum.(R); best = argmin(objs)
    for (i, r) in enumerate(R)
        a = at_sigma(m, Optim.minimizer(r)[1])
        row = [fmt(Optim.initial_x(r)[1]), fmt(Optim.minimizer(r)[1], d = 4), fsci(objs[i]),
               Optim.converged(r) ? "yes" : "no", string(Optim.iterations(r)), fsci(Optim.g_residual(r)), fsci(a.err)]
        push!(rows, i == best ? bold.(row) : row)
    end
    for f in failures
        push!(rows, [fmt(f.sigma0), "failed", "--", "--", "--", "--", "--"])
    end
    header = ["Start \$\\sigma_0\$", "\$\\hat\\sigma_\\alpha\$", "GMM obj.", "Converged", "Iterations", "\$\\|\\nabla\\|_\\infty\$", "Contraction error"]
    return tabular(header, rows; align = "rcccccc")
end

### First-stage helpers

In [ ]:
# HC1-robust OLS
function ols(y, X)
    n, k = size(X); b = X \ y; e = y - X * b
    XXi = inv(X' * X)
    V = XXi * (X' * ((e .^ 2) .* X)) * XXi * n / (n - k)
    return (b = b, se = sqrt.(diag(V)), r2 = 1 - sum(e .^ 2) / sum((y .- mean(y)) .^ 2), rss = sum(e .^ 2), n = n, k = k)
end

# First stage of y on X plus the instrument sets Z[:, s]; F tests the excluded instruments given X
function first_stage_table(y, X, Z, znames, sets, setnames)
    fits = [ols(y, hcat(X, Z[:, s])) for s in sets]; ols0 = ols(y, X)
    rows = Vector{Vector{String}}()
    for (lab, pos) in (("Constant", 1), ("\$x_2\$", 2), ("\$x_3\$", 3))
        push!(rows, vcat([lab], [est_cell(f.b[pos], f.se[pos]) for f in fits]))
        push!(rows, vcat([""], [se_cell(f.se[pos]) for f in fits]))
    end
    for (i, nm) in enumerate(znames)
        push!(rows, vcat([nm], [i in s ? est_cell(f.b[3 + findfirst(==(i), s)], f.se[3 + findfirst(==(i), s)]) : "" for (s, f) in zip(sets, fits)]))
        push!(rows, vcat([""], [i in s ? se_cell(f.se[3 + findfirst(==(i), s)]) : "" for (s, f) in zip(sets, fits)]))
    end
    nr = length(rows)
    push!(rows, vcat(["\$R^2\$"], [fmt(f.r2) for f in fits]))
    push!(rows, vcat(["\$F\$ (excluded instruments)"], [fmt(((ols0.rss - f.rss) / length(s)) / (f.rss / (f.n - f.k)), d = 1) for (s, f) in zip(sets, fits)]))
    push!(rows, vcat(["Observations"], [string(f.n) for f in fits]))
    return tabular(vcat(["Dependent variable: price"], setnames), rows; align = "l" * "c"^length(sets), rules = [nr])
end

## Q6 Market simulation and welfare (J=3 vs. J=5)

### Q6a Prices, markups and profits

In [ ]:
SRC6 = "true-parameter simulation of \\texttt{100markets3products.mat} and \\texttt{100markets5products.mat} with common consumer draws"
vars6a = [("prices", "Price", m -> vec(m.p)), ("markups", "Markup \$(p-c)/p\$", m -> vec(m.markup)), ("profits", "Profit", m -> vec(m.profit))]
for (nm, xl, f) in vars6a
    p = hist_kde([(f(truth_J3), "\$J=3\$", BLUE), (f(truth_J5), "\$J=5\$", RED)]; xlabel = xl, clipq = (nm == "prices" ? 0.01 : 0.05))
    save_fig(p, "q6a_hist_$(nm)"; q = "6a", caption = "Distribution of $(lowercase(split(xl)[1])), \$J=3\$ versus \$J=5\$ (tails beyond the " * (nm == "prices" ? "1st and 99th" : "5th and 95th") * " percentiles not drawn).", source = SRC6)
end
rows = Vector{Vector{String}}()
for (nm, xl, f) in vars6a, (jl, m) in (("3", truth_J3), ("5", truth_J5))
    push!(rows, vcat([split(xl)[1], jl], desc_cells(f(m))))
end
save_tab(tabular(vcat(["Variable", "\$J\$"], DESC_HEADER), rows; align = "llccccc", rules = [2, 4]), "q6a_desc_stats";
         q = "6a", caption = "Descriptive statistics of prices, markups and profits (all products and markets).", source = SRC6 * "; full sample, no trimming")

### Q6b Consumer surplus

In [ ]:
p = hist_kde([(vec(truth_J3.cs_sim), "\$J=3\$", BLUE), (vec(truth_J5.cs_sim), "\$J=5\$", RED)]; xlabel = "Consumer surplus (per market)")
save_fig(p, "q6b_hist_cs"; q = "6b", caption = "Distribution of consumer surplus across markets, \$J=3\$ versus \$J=5\$.", source = SRC6 * "; expected surplus averaged over the simulated consumers")
rows = [vcat(["3"], desc_cells(truth_J3.cs_sim)), vcat(["5"], desc_cells(truth_J5.cs_sim))]
save_tab(tabular(vcat(["\$J\$"], DESC_HEADER), rows; align = "lccccc"), "q6b_desc_cs";
         q = "6b", caption = "Descriptive statistics of consumer surplus across markets.", source = SRC6)

### Q6c Verification of simulated shares

In [ ]:
function share_sum_error(m)
    err = 0.0
    for t in 1:m.T
        idx = (t - 1) * m.J + 1:t * m.J
        out = mean(1 ./ (1 .+ sum(exp.(m.util[idx, :]), dims = 1)))      # simulated outside share
        err = max(err, abs(sum(m.shares_sim[idx]) + out - 1))
    end
    return err
end

diff3 = vec(truth_J3.shares - truth_J3.shares_sim); diff5 = vec(truth_J5.shares - truth_J5.shares_sim)
p = hist_kde([(diff3, "\$J=3\$", BLUE), (diff5, "\$J=5\$", RED)]; xlabel = "Real minus simulated share")
save_fig(p, "q6c_hist_share_diff"; q = "6c", caption = "Difference between real and simulated shares, \$J=3\$ versus \$J=5\$.", source = SRC6)

for (jl, m, c) in (("J3", truth_J3, BLUE), ("J5", truth_J5, RED))
    p = scat45([(vec(m.shares), vec(m.shares_sim), "\$J=" * jl[2:2] * "\$", c)]; xlabel = "Real share", ylabel = "Simulated share")
    save_fig(p, "q6c_scatter_shares_$(jl)"; q = "6c", caption = "Real versus simulated shares, \$J=" * jl[2:2] * "\$.", source = SRC6)
end

rows = Vector{Vector{String}}()
for (jl, m) in (("3", truth_J3), ("5", truth_J5))
    d = vec(m.shares - m.shares_sim)
    push!(rows, [jl, fmt(mean(d), d = 5), fmt(mean(abs.(d)), d = 5), fmt(sqrt(mean(d .^ 2)), d = 5), fmt(cor(vec(m.shares), vec(m.shares_sim)), d = 5), fsci(share_sum_error(m))])
end
save_tab(tabular(["\$J\$", "Mean diff.", "Mean abs. diff.", "RMSE", "Correlation", "Max \$|\\sum_j s_{jt}+s_{0t}-1|\$"], rows; align = "lccccc"), "q6c_share_discrepancy";
         q = "6c", caption = "Discrepancy between real and simulated shares (real minus simulated), and maximum deviation of simulated shares plus outside share from one.", source = SRC6)

## Q7 Demand moments and instruments (T=100, J=3)

### Q7a

In [ ]:
price_bar_J3 = blp.compute_hausman(truth_J3)
X_rivals_J3  = blp.compute_blp(truth_J3)
xi_J3 = vec(truth_J3.xi_real)

# Per-observation product (xi * z); H0: its mean is zero. t = mean / (sd / sqrt(N)), two-sided p from t(N-1).
function moment_row(label, z, xi)
    g = xi .* vec(z); n = length(g)
    t = mean(g) / (std(g) / sqrt(n))
    return [label, fmt(mean(g), d = 4), fmt(std(g), d = 4), fmt(t, d = 3), fmt(2 * (1 - cdf(TDist(n - 1), abs(t))), d = 3), string(n)]
end
HDR7 = ["Moment", "Mean", "SD", "\$t\$-stat", "\$p\$-value", "\$N\$"]
SRC7 = "true structural errors \$\\xi_{jt}\$ and data of \\texttt{100markets3products.mat}"

rows7a = [moment_row("\$E[\\xi_{jt}\\cdot 1]\$", ones(length(xi_J3)), xi_J3),
          moment_row("\$E[\\xi_{jt}x_{2jt}]\$", truth_J3.X[:, 2], xi_J3),
          moment_row("\$E[\\xi_{jt}x_{3jt}]\$", truth_J3.X[:, 3], xi_J3),
          moment_row("\$E[\\xi_{jt}p_{jt}]\$", truth_J3.p, xi_J3),
          moment_row("\$E[\\xi_{jt}\\bar p_{j,-t}]\$", price_bar_J3, xi_J3)]
save_tab(tabular(HDR7, rows7a; align = "lccccc"), "q7a_moments"; q = "7a",
         caption = "Sample analogues of the moments \$E[\\xi_{jt}z_{jt}]\$ and \$t\$-tests of \$H_0:E[\\xi_{jt}z_{jt}]=0\$.", source = SRC7)

### Q7c

In [ ]:
rows7c = [moment_row("\$E[\\xi_{jt}\\sum_{k\\neq j}x_{2kt}]\$", X_rivals_J3[:, 1], xi_J3),
          moment_row("\$E[\\xi_{jt}\\sum_{k\\neq j}x_{3kt}]\$", X_rivals_J3[:, 2], xi_J3)]
save_tab(tabular(HDR7, rows7c; align = "lccccc"), "q7c_moments"; q = "7c",
         caption = "Sample analogues of the moments with the BLP rival-characteristic sums and \$t\$-tests of \$H_0:E[\\xi_{jt}z_{jt}]=0\$.", source = SRC7)

## Q8 Mixed logit by nested fixed point

### Q8a Mean utilities and individual utility component (true parameters, T=100, J=3)

In [ ]:
SRC8a = "true-parameter simulation of \\texttt{100markets3products.mat}; \$\\delta_{jt}=x_{jt}'\\beta-\\bar\\alpha p_{jt}+\\xi_{jt}\$, \$\\mu_{ijt}=-\\sigma_\\alpha\\nu_i p_{jt}\$"
delta_true = vec(truth_J3.delta)
mu_true = vec(truth_J3.util .- truth_J3.delta)
save_fig(hist_kde([(delta_true, "\$\\delta_{jt}\$", BLUE)]; xlabel = "Mean utility \$\\delta_{jt}\$"), "q8a_hist_delta"; q = "8a", caption = "Distribution of mean utilities \$\\delta_{jt}\$ at the true parameters.", source = SRC8a)
save_fig(hist_kde([(mu_true, "\$\\mu_{ijt}\$", RED)]; xlabel = "Individual utility component \$\\mu_{ijt}\$", clipq = 0.01), "q8a_hist_mu"; q = "8a", caption = "Distribution of \$\\mu_{ijt}\$ over products, markets and simulated consumers at the true parameters (tails beyond the 1st and 99th percentiles not drawn).", source = SRC8a)

### Q8c Estimates by starting point (T=100, J=3, BLP instruments only)

In [ ]:
SRC8c = "author's calculations from the stored full estimates \\texttt{q8c\\_blp\\_only\\_T100\\_J3} (per-start values recomputed at each stored \$\\sigma\$ minimizer, no re-optimization)"
NOTE_BOLD = " Bold: start with the lowest GMM objective across starts."
save_tab(by_start_table(m8c_only), "q8c_estimates_by_start"; q = "8c", caption = "Estimates by starting point, T=100, J=3, BLP instruments (standard errors in parentheses; * \$p<0.1\$, ** \$p<0.05\$, *** \$p<0.01\$).", source = SRC8c * "." * NOTE_BOLD * NOTE_NA)
save_tab(convergence_table(m8c_only, f8c_only), "q8c_convergence_by_start"; q = "8c", caption = "Convergence criterion and contraction error by starting point, T=100, J=3.", source = SRC8c * "; failed starts from \\texttt{q8c\\_blp\\_only\\_T100\\_J3\\_failed\\_starts}." * NOTE_BOLD)
save_tab(reg_table(["BLP", "BLP + Hausman"], [m8c_only, m8c_haus]), "q8c_hausman_compare"; q = "8c", caption = "Estimates with and without the Hausman instrument, T=100, J=3.", source = "author's calculations from the stored full estimates \\texttt{q8c\\_blp\\_only\\_T100\\_J3} and \\texttt{q8c\\_blp\\_hausman\\_T100\\_J3}; best start of each." * NOTE_NA)

# First stage of price on the instruments (T=100, J=3): BLP rival sums, and BLP rival sums + Hausman
Z8c = hcat(X_rivals_J3, price_bar_J3)          # BLP rival x2, x3 | Hausman
save_tab(first_stage_table(vec(truth_J3.p), truth_J3.X, Z8c, ["BLP rival sum \$x_2\$", "BLP rival sum \$x_3\$", "Hausman \$\\bar p_{j,-t}\$"], [[1, 2], [1, 2, 3]], ["BLP", "BLP + Hausman"]), "q8c_first_stage";
         q = "8c", caption = "First stage of price on the instruments, T=100, J=3 (heteroskedasticity-robust standard errors in parentheses; * \$p<0.1\$, ** \$p<0.05\$, *** \$p<0.01\$; \$F\$ tests the excluded instruments given \$x\$).",
         source = "author's calculations from \\texttt{100markets3products.mat}; regressions include a constant, \$x_2\$ and \$x_3\$")

### Q8d Elasticities at the estimates versus the truth (T=100, J=3, model of 8c)

In [ ]:
SRC8d = "author's calculations: elasticities at the estimates of \\texttt{q8c\\_blp\\_only\\_T100\\_J3} versus the true-parameter simulation, \\texttt{100markets3products.mat}"
E_true = truth_J3.elas_sim; E_est = m8c_only.elas_sim
p = scat45([(E_true, E_est, "Simulated (estimates)", RED)]; xlabel = "Real elasticity (true parameters)", ylabel = "Simulated elasticity (estimates)", line_label = "Real (45° line)")
save_fig(p, "q8d_scatter_elas"; q = "8d", caption = "Price elasticities at the estimates versus at the true parameters (all own- and cross-price elements).", source = SRC8d)
# One row per element (j,k) of the elasticity matrix: mean and SD of real, simulated and the difference across markets
rows = Vector{Vector{String}}()
for j in 1:3, k in 1:3
    r = E_true[j:3:end, k]; s = E_est[j:3:end, k]
    push!(rows, vcat(["\$($j,$k)\$"], [fmt(f(x)) for x in (r, s, s .- r) for f in (mean, std)]))
end
L = ["\\begin{tabular}{lcccccc}", "\\toprule",
     " & \\multicolumn{2}{c}{Real} & \\multicolumn{2}{c}{Simulated} & \\multicolumn{2}{c}{Simulated \$-\$ real} \\\\",
     "\\cmidrule(lr){2-3}\\cmidrule(lr){4-5}\\cmidrule(lr){6-7}",
     "Element \$(j,k)\$ & Mean & SD & Mean & SD & Mean & SD \\\\", "\\midrule"]
for r in rows; push!(L, join(r, " & ") * " \\\\"); end
push!(L, "\\bottomrule", "\\end{tabular}")
save_tab(L, "q8d_elas_cells"; q = "8d",
         caption = "Elasticity of the share of product \$j\$ with respect to the price of product \$k\$: real versus simulated (at the estimates), mean and standard deviation across markets.", source = SRC8d)

### Q8e Same outputs for T=10, J=3

In [ ]:
SRC8e = "author's calculations from the stored full estimates \\texttt{q8e\\_blp\\_only\\_T10\\_J3} (per-start values recomputed at each stored \$\\sigma\$ minimizer, no re-optimization)"
save_tab(by_start_table(m8e_only), "q8e_estimates_by_start"; q = "8e", caption = "Estimates by starting point, T=10, J=3, BLP instruments (standard errors in parentheses; * \$p<0.1\$, ** \$p<0.05\$, *** \$p<0.01\$).", source = SRC8e * "." * NOTE_BOLD * NOTE_NA)
save_tab(convergence_table(m8e_only, f8e_only), "q8e_convergence_by_start"; q = "8e", caption = "Convergence criterion and contraction error by starting point, T=10, J=3.", source = SRC8e * "; failed starts from \\texttt{q8e\\_blp\\_only\\_T10\\_J3\\_failed\\_starts}." * NOTE_BOLD)
save_tab(reg_table(["BLP", "BLP + Hausman"], [m8e_only, m8e_haus]), "q8e_hausman_compare"; q = "8e", caption = "Estimates with and without the Hausman instrument, T=10, J=3.", source = "author's calculations from the stored full estimates \\texttt{q8e\\_blp\\_only\\_T10\\_J3} and \\texttt{q8e\\_blp\\_hausman\\_T10\\_J3}; best start of each." * NOTE_NA)

### Q8f Price as its own instrument

In [ ]:
save_tab(reg_table(["BLP (IV)", "Price as instrument"], [m8c_only, m8f_price]), "q8f_compare_original"; q = "8f",
         caption = "Estimates imposing \$E[\\xi_{jt}p_{jt}]=0\$ (price as its own instrument) versus the original IV specification.",
         source = "author's calculations from the stored full estimates \\texttt{q8c\\_blp\\_only\\_T100\\_J3} (original) and \\texttt{q8f\\_blp\\_price\\_T100\\_J3}")
save_tab(reg_table(["Price as instrument", "Price + Hausman"], [m8f_price, m8f_price_h]), "q8f_hausman_compare"; q = "8f",
         caption = "Price-as-instrument estimates with and without the Hausman instrument.",
         source = "author's calculations from the stored full estimates \\texttt{q8f\\_blp\\_price\\_T100\\_J3} and \\texttt{q8f\\_blp\\_price\\_hausman\\_T100\\_J3}")

## Q9 Cost shifters and the supply side

### Q9a Adding the cost shifter w (T=100, J=3)

In [ ]:
SRC9a = "author's calculations from the stored full estimates \\texttt{q8c\\_*} and \\texttt{q9a\\_*} (T=100, J=3)"
save_tab(reg_table(["BLP (Q8)", "BLP \$+\\,w\$ (Q9a)", "BLP + Haus. (Q8)", "BLP + Haus. \$+\\,w\$ (Q9a)"], [m8c_only, m9a_w, m8c_haus, m9a_haus_w]), "q9a_estimates";
         q = "9a", caption = "Demand estimates with and without the cost shifter \$w_j\$ in the instrument set (standard errors in parentheses; * \$p<0.1\$, ** \$p<0.05\$, *** \$p<0.01\$).", source = SRC9a)

profit_true = vec(truth_J3.profit)
p = scat45([(profit_true, vec(m8c_only.profit_sim), "Without \$w\$ (Q8)", BLUE), (profit_true, vec(m9a_w.profit_sim), "With \$w\$ (Q9a)", RED)];
           xlabel = "Real profit (true costs)", ylabel = "Profit at the estimates")
save_fig(p, "q9a_scatter_profits"; q = "9a", caption = "Product-level profits at the estimates, with and without the cost shifter, versus real profits.", source = SRC9a * "; profits \$(p-\\hat c)s\$ with costs recovered under Bertrand pricing")

own(E) = vcat([diag(E[(t - 1) * 3 + 1:t * 3, :]) for t in 1:(size(E, 1) ÷ 3)]...)
p = scat45([(own(truth_J3.elas_sim), own(m8c_only.elas_sim), "Without \$w\$ (Q8)", BLUE), (own(truth_J3.elas_sim), own(m9a_w.elas_sim), "With \$w\$ (Q9a)", RED)];
           xlabel = "Real own-price elasticity", ylabel = "Own-price elasticity at the estimates", legend = :bottomright)
save_fig(p, "q9a_scatter_elas"; q = "9a", caption = "Own-price elasticities at the estimates, with and without the cost shifter, versus real elasticities.", source = SRC9a)

### Q9b Small sample (T=10, J=3)

In [ ]:
save_tab(reg_table(["BLP (Q8e)", "BLP \$+\\,w\$ (Q9b)", "BLP + Haus. (Q8e)", "BLP + Haus. \$+\\,w\$ (Q9b)"], [m8e_only, m9b_w, m8e_haus, m9b_haus_w]), "q9b_estimates";
         q = "9b", caption = "Small-sample estimates (T=10, J=3) with and without the cost shifter \$w_j\$ (standard errors in parentheses; * \$p<0.1\$, ** \$p<0.05\$, *** \$p<0.01\$).",
         source = "author's calculations from the stored full estimates \\texttt{q8e\\_*} and \\texttt{q9b\\_*} (T=10, J=3)")

### Q9d Recovered marginal costs under three conduct assumptions

In [ ]:
# Demand estimates of 9a with Hausman and w; only the ownership matrix changes the cost inversion c = p - (Omega .* Delta)^-1 s
function recover_cost(m, Om)
    mm = deepcopy(m); mm.Omega = Om
    blp.simulate_BLP!(mm, all = true)
    return vec(mm.cost_sim)
end
c_true = vec(truth_J3.cost)
c_comp = recover_cost(m9a_haus_w, zeros(3, 3)); c_bert = recover_cost(m9a_haus_w, Matrix{Float64}(I, 3, 3)); c_coll = recover_cost(m9a_haus_w, ones(3, 3))
p = scat45([(c_true, c_comp, "Perfect competition", LBLUE), (c_true, c_bert, "Bertrand", DBLUE), (c_true, c_coll, "Perfect collusion", RED)];
           xlabel = "True marginal cost", ylabel = "Recovered marginal cost", legend = :topleft, qlim = 0.01)
save_fig(p, "q9d_scatter_cost"; q = "9d", caption = "True versus recovered marginal costs under perfect competition, Bertrand pricing and perfect collusion (axes cover the 1st to 99th percentile of the plotted values).",
         source = "author's calculations: cost inversion at the demand estimates of \\texttt{q9a\\_blp\\_hausman\\_w\\_T100\\_J3}")

### Q9e Joint demand and supply estimation

In [ ]:
SRC9e = "author's calculations from the stored full estimates \\texttt{q9e\\_system\\_*\\_T100\\_J3}"
save_tab(reg_table(["Bertrand", "Competition", "Collusion"], [m9e_bert, m9e_comp, m9e_coll]), "q9e_estimates";
         q = "9e", caption = "Joint demand and supply estimates under three conduct assumptions (standard errors in parentheses; * \$p<0.1\$, ** \$p<0.05\$, *** \$p<0.01\$).", source = SRC9e)

sys3 = [("Bertrand", m9e_bert, DBLUE), ("Competition", m9e_comp, LBLUE), ("Collusion", m9e_coll, RED)]
Et = truth_J3.elas_sim
p = scat45([(Et, dem(m).elas_sim, l, c) for (l, m, c) in sys3]; xlabel = "Real elasticity", ylabel = "Simulated elasticity", ms = 3)
save_fig(p, "q9e_scatter_elas"; q = "9e", caption = "Real versus simulated price elasticities (all own- and cross-price elements) under the three conduct assumptions.", source = SRC9e)
ownE(E) = own(E); crossE(E) = [E[(t - 1) * 3 + j, k] for t in 1:100 for j in 1:3 for k in 1:3 if j != k]
blk(title, f) = vcat([["\\multicolumn{6}{l}{\\textit{$title}}"]],
                     [vcat(["Real"], desc_cells(f(Et)))], [vcat([l], desc_cells(f(dem(m).elas_sim))) for (l, m, c) in sys3])
rows = vcat(blk("Own-price elasticities", ownE), blk("Cross-price elasticities", crossE))
save_tab(tabular(vcat([""], DESC_HEADER), rows; align = "lccccc", rules = [5]), "q9e_elas_desc"; q = "9e", caption = "Descriptive statistics of real and simulated elasticities.", source = SRC9e)

Mt = vec(truth_J3.markup)
p = scat45([(Mt, vec(dem(m).markup_sim), l, c) for (l, m, c) in sys3]; xlabel = "Real markup \$(p-c)/p\$", ylabel = "Simulated markup", ms = 3, qlim = 0.01)
save_fig(p, "q9e_scatter_markup"; q = "9e", caption = "Real versus simulated markups under the three conduct assumptions (axes cover the 1st to 99th percentile of the plotted values).", source = SRC9e)
rows = vcat([vcat(["Real"], desc_cells(Mt))], [vcat([l], desc_cells(vec(dem(m).markup_sim))) for (l, m, c) in sys3])
save_tab(tabular(vcat([""], DESC_HEADER), rows; align = "lccccc"), "q9e_markup_desc"; q = "9e", caption = "Descriptive statistics of real and simulated markups.", source = SRC9e)

## Q10 Merger simulation (firms 1 and 2, Bertrand system of 9e)

### Q10b Price and surplus changes

In [ ]:
SRC10 = "author's calculations: merger counterfactual (firms 1 and 2, no cost efficiencies) at the estimates of \\texttt{q9e\\_system\\_bertrand\\_T100\\_J3}"
pre = deepcopy(m9e_bert); post = deepcopy(m9e_bert)
post.BLPDemand.Omega = [1.0 1.0 0.0; 1.0 1.0 0.0; 0.0 0.0 1.0]
post = blp.counterfactual_simulation(post)
dpre, dpost = pre.BLPDemand, post.BLPDemand
J10, T10 = dpre.J, dpre.T
@assert all(isfinite, dpost.cs_sim) "counterfactual did not converge"

# Producer surplus with the pre-merger marginal costs (no efficiencies) and simulated shares in both states
ps_of(d, c) = vec(sum(reshape((d.p - c) .* d.shares_sim, J10, T10), dims = 1))
dCS = vec(dpost.cs_sim - dpre.cs_sim)
dPS = ps_of(dpost, dpre.cost_sim) - ps_of(dpre, dpre.cost_sim)
dTS = dCS + dPS
dP = reshape(vec(dpost.p - dpre.p), J10, T10)          # product x market

p = hist_kde([(dP[1, :], "Product 1", BLUE), (dP[2, :], "Product 2", RED), (dP[3, :], "Product 3", LBLUE)]; xlabel = "Post-merger price change \$\\Delta p_j\$", nbins = 25)
save_fig(p, "q10b_hist_dp"; q = "10b", caption = "Distribution and kernel density of the post-merger price changes of products 1, 2 (merging) and 3 (non-merging) across markets.", source = SRC10)
save_tab(tabular(vcat(["Product"], DESC_HEADER), [vcat([string(j)], desc_cells(dP[j, :])) for j in 1:3]; align = "lccccc"), "q10b_dp_desc";
         q = "10b", caption = "Descriptive statistics of post-merger price changes by product.", source = SRC10)
save_tab(tabular(vcat([""], DESC_HEADER), [vcat(["\$\\Delta\$ Consumer surplus"], desc_cells(dCS)), vcat(["\$\\Delta\$ Producer surplus"], desc_cells(dPS)), vcat(["\$\\Delta\$ Total surplus"], desc_cells(dTS))]; align = "lccccc"), "q10b_dsurplus_desc";
         q = "10b", caption = "Descriptive statistics of the post-merger changes in consumer, producer and total surplus across markets.", source = SRC10)

### Q10c Linear approximations (diversion and upward pricing pressure)

In [ ]:
idx = J10 * (0:T10 - 1)
D12 = dpre.ds_dp[1 .+ idx, 2] ./ -dpre.ds_dp[1 .+ idx, 1]        # diversion 1 -> 2
D21 = dpre.ds_dp[1 .+ idx, 2] ./ -dpre.ds_dp[2 .+ idx, 2]        # diversion 2 -> 1
pc1 = (dpre.p - dpre.cost_sim)[1 .+ idx]; pc2 = (dpre.p - dpre.cost_sim)[2 .+ idx]
s1 = dpre.shares_sim[1 .+ idx]; s2 = dpre.shares_sim[2 .+ idx]; p1 = dpre.p[1 .+ idx]
UPP1 = D12 .* pc2; UPP2 = D21 .* pc1
# Linear approximations as in pset_io (cells 145-146)
cs_lin = -(UPP1 .* s1 + UPP2 .* s2)
ps_lin = (UPP1 .^ 2) .* s1 ./ p1 + (UPP2 .^ 2) .* s2
ts_lin = cs_lin + ps_lin

lin = [("dp1", UPP1, dP[1, :], "Linear approximation of \$\\Delta p_1\$ (UPP\$_1\$)", "Simulated \$\\Delta p_1\$", "Price change of product 1", "price change of product 1 against upward pricing pressure"),
       ("dp2", UPP2, dP[2, :], "Linear approximation of \$\\Delta p_2\$ (UPP\$_2\$)", "Simulated \$\\Delta p_2\$", "Price change of product 2", "price change of product 2 against upward pricing pressure"),
       ("dcs", cs_lin, dCS, "Linear approximation of \$\\Delta CS\$", "Simulated \$\\Delta CS\$", "Change in consumer surplus", "change in consumer surplus against its linear approximation"),
       ("dps", ps_lin, dPS, "Linear approximation of \$\\Delta PS\$", "Simulated \$\\Delta PS\$", "Change in producer surplus", "change in producer surplus against its linear approximation"),
       ("dts", ts_lin, dTS, "Linear approximation of \$\\Delta TS\$", "Simulated \$\\Delta TS\$", "Change in total surplus", "change in total surplus against its linear approximation")]
for (nm, x, y, xl, yl, _, cap) in lin
    p = scat45([(x, y, "Simulated (markets)", RED)]; xlabel = xl, ylabel = yl, line_label = "Linear approximation (45\u00b0 line)")
    save_fig(p, "q10c_scatter_$(nm)"; q = "10c", caption = "Simulated $cap, one point per market.", source = SRC10 * "; diversion ratios and \$\\mathrm{UPP}_j=D_{jk}(p_k-c_k)\$ from the pre-merger estimates")
end

## Q11 Newer instruments and substitution diagnostics (T=100, J=5)

### Q11a Instruments, first stage, estimates and sensitivity

In [ ]:
J5, T5 = truth_J5.J, truth_J5.T
X5 = truth_J5.X
haus5 = blp.compute_hausman(truth_J5)
blp5  = blp.compute_blp(truth_J5)
gh1 = zeros(T5 * J5, 2); gh2 = zeros(T5 * J5, 2)          # columns: x2, x3
for col in 2:3
    cutoff = std(X5[:, col])
    for t in 1:T5
        rows_ = (t - 1) * J5 + 1:t * J5
        f = X5[rows_, col]
        gh1[rows_, col - 1] = sum(abs.(f .- f') .< cutoff, dims = 2) .- 1
        gh2[rows_, col - 1] = sum(abs.(f .- f') .^ 2, dims = 2)
    end
end
ZALL = hcat(haus5, blp5, gh1, gh2)         # Hausman | BLP x2, x3 | GH local x2, x3 | GH quad x2, x3
ZNAMES = ["Hausman \$\\bar p_{j,-t}\$", "BLP rival sum \$x_2\$", "BLP rival sum \$x_3\$", "GH local rival \$x_2\$", "GH local rival \$x_3\$", "GH quadratic rival \$x_2\$", "GH quadratic rival \$x_3\$"]
sets = [[1], [2, 3], [4, 5], [6, 7], collect(1:7)]
setnames = ["Hausman", "BLP", "GH local", "GH quadratic", "All"]
y5 = vec(truth_J5.p)
ols0 = ols(y5, X5)
fits = [ols(y5, hcat(X5, ZALL[:, s])) for s in sets]

rows = Vector{Vector{String}}()
for (lab, pos) in (("Constant", 1), ("\$x_2\$", 2), ("\$x_3\$", 3))
    push!(rows, vcat([lab], [est_cell(f.b[pos], f.se[pos]) for f in fits]))
    push!(rows, vcat([""], [se_cell(f.se[pos]) for f in fits]))
end
for (i, nm) in enumerate(ZNAMES)
    push!(rows, vcat([nm], [i in s ? est_cell(f.b[3 + findfirst(==(i), s)], f.se[3 + findfirst(==(i), s)]) : "" for (s, f) in zip(sets, fits)]))
    push!(rows, vcat([""], [i in s ? se_cell(f.se[3 + findfirst(==(i), s)]) : "" for (s, f) in zip(sets, fits)]))
end
nr = length(rows)
push!(rows, vcat(["\$R^2\$"], [fmt(f.r2) for f in fits]))
push!(rows, vcat(["\$F\$ (excluded instruments)"], [fmt(((ols0.rss - f.rss) / length(s)) / (f.rss / (f.n - f.k)), d = 1) for (s, f) in zip(sets, fits)]))
push!(rows, vcat(["Observations"], [string(f.n) for f in fits]))
save_tab(tabular(vcat(["Dependent variable: price"], setnames), rows; align = "lccccc", rules = [nr]), "q11a_first_stage";
         q = "11a", caption = "Price first stage with each instrument family, T=100, J=5 (heteroskedasticity-robust standard errors in parentheses; \$F\$ tests the excluded instruments given \$x\$).",
         source = "author's calculations from \\texttt{100markets5products.mat}; first-stage regressions include a constant, \$x_2\$ and \$x_3\$")

In [ ]:
SRC11 = "author's calculations from the stored full estimates \\texttt{q11a\\_*\\_T100\\_J5}"
save_tab(reg_table(["BLP", "GH", "GH + Hausman"], [m11_blp, m11_gh, m11_ghh]), "q11a_estimates_all";
         q = "11a", caption = "Estimates and final GMM objective by instrument set, T=100, J=5 (standard errors in parentheses; * \$p<0.1\$, ** \$p<0.05\$, *** \$p<0.01\$).", source = SRC11 * "; best start of each. No fixed effects are used in any specification." * NOTE_NA)

function sens_row(label, m, fails)
    R = m.opt_res
    s = [Optim.minimizer(r)[1] for r in R]; o = Optim.minimum.(R)
    return [label, string(length(R)), string(length(fails)), string(count(Optim.converged, R)), fmt(minimum(s), d = 4), fmt(maximum(s), d = 4), fsci(minimum(o)), fsci(maximum(o)), fmt(mean(Optim.iterations.(R)), d = 1)]
end
rows = [sens_row("BLP", m11_blp, f11_blp), sens_row("GH", m11_gh, f11_gh), sens_row("GH + Hausman", m11_ghh, f11_ghh)]
save_tab(tabular(["Instruments", "Starts ok", "Starts failed", "Converged", "Min \$\\hat\\sigma_\\alpha\$", "Max \$\\hat\\sigma_\\alpha\$", "Min obj.", "Max obj.", "Mean iter."], rows; align = "lcccccccc"), "q11a_sensitivity";
         q = "11a", caption = "Sensitivity to starting values, T=100, J=5.", source = SRC11 * " and the corresponding \\texttt{*\\_failed\\_starts} files; summary over all starting points")

### Q11b Variation left after market and product fixed effects

In [ ]:
# Balanced panel (every market has all J products), so the two-way fixed-effects residual is the double demeaning
# x_jt - xbar_j. - xbar_.t + xbar; the one-way ones are the single demeanings.
function fe_resid(x, J, T)
    M = reshape(vec(x), J, T)
    mj = mean(M, dims = 2); mt = mean(M, dims = 1)
    return (market = vec(M .- mt), product = vec(M .- mj), both = vec(M .- mj .- mt .+ mean(M)))
end
r2_of(x, e) = 1 - sum(abs2, e) / sum(abs2, x .- mean(x))

vars11b = vcat([("Price \$p_{jt}\$", y5)], [(ZNAMES[i], ZALL[:, i]) for i in 1:7])
rows = Vector{Vector{String}}()
for (nm, x) in vars11b
    e = fe_resid(x, J5, T5)
    push!(rows, [nm, fmt(std(x)), fmt(r2_of(x, e.market)), fmt(r2_of(x, e.product)), fmt(r2_of(x, e.both)), fmt(std(e.both)), fmt(1 - r2_of(x, e.both))])
end
save_tab(tabular(["Variable", "SD", "\$R^2\$ market FE", "\$R^2\$ product FE", "\$R^2\$ both", "SD after both", "Share of variance left"], rows; align = "lcccccc", rules = [1]), "q11b_fe_variation";
         q = "11b", caption = "Variation of the price and of each instrument explained by market and product fixed effects, T=100, J=5; the last two columns show what remains after both sets of effects.",
         source = "author's calculations from \\texttt{100markets5products.mat}; \$R^2\$ from regressing each variable on market dummies, product dummies, or both (balanced panel)")

### Q11c Semi-elasticities against price (GH + Hausman estimates)

In [ ]:
m = m11_ghh
eta = [-m.ds_dp[(t - 1) * J5 + j, j] / m.shares_sim[(t - 1) * J5 + j] for t in 1:T5 for j in 1:J5]   # own-price semi-elasticity
pr = vec(m.p); x2 = m.X[:, 2]; x3 = m.X[:, 3]
# Small-choice-probability bound of the slides: i_scp / |eta| small means the approximation error is small
i_scp = vec((mean(m.choice .^ 4, dims = 2) ./ (mean(m.choice, dims = 2) .^ 2) .* mean((m.beta[end] .- exp.(m.errors .* m.sigma)) .^ 2)) .^ 0.5)
ratio = abs.(i_scp ./ eta)
keep = ratio .< 0.25
SRC11c = "author's calculations at the stored estimates \\texttt{q11a\\_gh\\_hausman\\_T100\\_J5}; \$\\eta_{jt}=-s_{jt}^{-1}\\partial s_{jt}/\\partial p_{jt}\$"
p = plot(; xlabel = ptxt("Price \$p_{jt}\$"), ylabel = ptxt("Own-price semi-elasticity \$\\eta_{jt}\$"), legend = :topleft)
scatter!(p, pr, eta; color = RED, ms = 3.5, alpha = 0.6, markerstrokewidth = 0, label = "J = 5, all products")
save_fig(p, "q11c_scatter_semielas"; q = "11c", caption = "Own-price semi-elasticity against price, all product-markets.", source = SRC11c)
p = plot(; xlabel = ptxt("Price \$p_{jt}\$"), ylabel = ptxt("Own-price semi-elasticity \$\\eta_{jt}\$"), legend = :topleft)
scatter!(p, pr[keep], eta[keep]; color = RED, ms = 3.5, alpha = 0.6, markerstrokewidth = 0, label = "J = 5, restricted sample")
save_fig(p, "q11c_scatter_semielas_restricted"; q = "11c", caption = "Own-price semi-elasticity against price, restricted sample with small approximation error (\$|i^{scp}_{jt}/\\eta_{jt}|<0.25\$).", source = SRC11c * "; approximation-error ratio as in \\texttt{pset\\_io}")

# Same flexible specification in every column; only the sample changes: the smaller the cut-off on |i_scp/eta|, the more restricted the sample
function flex(idx)
    y = eta[idx]; P = pr[idx]; A = x2[idx]; B = x3[idx]; o = ones(length(y))
    return ols(y, hcat(o, P, A, B, P .^ 2, 1 ./ P, A .^ 2, B .^ 2, P .* A, P .* B))
end
cuts = [Inf, 2.0, 1.0, 0.75, 0.5, 0.25, 0.1]
fits11c = [flex(ratio .< c) for c in cuts]
labs = ["Constant", "\$p\$", "\$x_2\$", "\$x_3\$", "\$p^2\$", "\$1/p\$", "\$x_2^2\$", "\$x_3^2\$", "\$p\\cdot x_2\$", "\$p\\cdot x_3\$"]
rows = Vector{Vector{String}}()
for i in 1:10
    push!(rows, vcat([labs[i]], [est_cell(f.b[i], f.se[i]) for f in fits11c]))
    push!(rows, vcat([""], [se_cell(f.se[i]) for f in fits11c]))
end
nr = length(rows)
push!(rows, vcat(["\$R^2\$"], [fmt(f.r2) for f in fits11c]))
push!(rows, vcat(["Observations"], [string(f.n) for f in fits11c]))
hdr = vcat(["Cut-off on \$|i^{scp}_{jt}/\\eta_{jt}|\$"], [isinf(c) ? "Full" : fmt(c, d = 2) for c in cuts])
save_tab(tabular(hdr, rows; align = "lccccccc", rules = [nr]), "q11c_regressions";
         q = "11c", caption = "Flexible regression of the own-price semi-elasticity on price and characteristics (the same specification in every column). Columns differ only in the sample: the cut-off on \$|i^{scp}_{jt}/\\eta_{jt}|\$ falls from left to right, so the sample becomes more restricted (heteroskedasticity-robust standard errors in parentheses; * \$p<0.1\$, ** \$p<0.05\$, *** \$p<0.01\$).",
         source = SRC11c * "; each column keeps the product-markets with \$|i^{scp}_{jt}/\\eta_{jt}|\$ below the cut-off in the header")

### Q11d Effective rank of the model-implied diversion matrix (mixed logit only)

In [ ]:
# D = diag(s)^-1 sum_i pi_i s~_i s_i',  s~_i = s_i / (1 - s_i), uniform weights over the R simulated consumers
function erank_market(ch)
    R = size(ch, 2)
    s = vec(mean(ch, dims = 2))
    D = (Diagonal(1 ./ s) * ((ch ./ (1 .- ch)) * ch') ./ R)
    sv = svdvals(D)
    q = sv ./ sum(sv)
    return exp(-sum(q .* log.(q .+ 1e-300)))
end
erank = [erank_market(m.choice[(t - 1) * J5 + 1:t * J5, :]) for t in 1:T5]
k = kde(erank)
p = plot(k.x, k.density; color = RED, lw = 2.5, fillrange = 0, fillalpha = 0.25, fillcolor = RED, label = "J = 5, mixed logit", xlabel = "Effective rank of the diversion matrix", ylabel = "Density", legend = :topright)
save_fig(p, "q11d_density_erank"; q = "11d", caption = "Density of the effective rank (exponential of the entropy of normalized singular values) of the model-implied diversion matrix across markets.",
         source = "author's calculations at the stored estimates \\texttt{q11a\\_gh\\_hausman\\_T100\\_J5}; mixed logit only")

## Overleaf skeleton

In [ ]:
# Figures shown together as subfigures: (id, member files, group caption, panel captions)
const GROUPS = [
    ("grp_q6a", ["q6a_hist_prices", "q6a_hist_markups", "q6a_hist_profits"], "Prices, markups and profits, \$J=3\$ versus \$J=5\$ (tails not drawn).", ["Prices", "Markups", "Profits"]),
    ("grp_q6c", ["q6c_hist_share_diff", "q6c_scatter_shares_J3", "q6c_scatter_shares_J5"], "Real versus simulated shares.", ["Difference, real minus simulated", "Real versus simulated, \$J=3\$", "Real versus simulated, \$J=5\$"]),
    ("grp_q8a", ["q8a_hist_delta", "q8a_hist_mu"], "Mean utility and individual utility component at the true parameters.", ["Mean utility \$\\delta_{jt}\$", "Individual component \$\\mu_{ijt}\$ (tails not drawn)"]),
    ("grp_q9a", ["q9a_scatter_profits", "q9a_scatter_elas"], "Profits and own-price elasticities at the estimates, with and without the cost shifter, against real values.", ["Profits", "Own-price elasticities"]),
    ("grp_q9e", ["q9e_scatter_elas", "q9e_scatter_markup"], "Real versus simulated elasticities and markups under the three conduct assumptions.", ["Elasticities", "Markups (axes cover the 1st to 99th percentile)"]),
    ("grp_q10c", ["q10c_scatter_dp1", "q10c_scatter_dp2", "q10c_scatter_dcs", "q10c_scatter_dps", "q10c_scatter_dts"], "Simulated merger effects against their linear (upward pricing pressure) approximations, one point per market.", ["Price change of product 1", "Price change of product 2", "Change in consumer surplus", "Change in producer surplus", "Change in total surplus"]),
    ("grp_q11c", ["q11c_scatter_semielas", "q11c_scatter_semielas_restricted"], "Own-price semi-elasticity against price.", ["All product-markets", "Restricted sample (\$|i^{scp}_{jt}/\\eta_{jt}|<0.25\$)"]),
]
group_of(name) = findfirst(g -> name in g[2], GROUPS)

function group_block(g)
    id, members, cap, pcaps = g
    its = [first(filter(i -> i.name == m, ITEMS)) for m in members]
    srcs = unique([it.source for it in its])
    L = ["\\begin{figure}[H]\\centering\\caption{$cap}\\label{fig:$id}"]
    for (k, (it, pc)) in enumerate(zip(its, pcaps))
        push!(L, "\\begin{subfigure}[t]{0.48\\textwidth}\\centering\\includegraphics[width=\\linewidth]{$(it.name).pdf}\\caption{$pc}\\label{fig:$(it.name)}\\end{subfigure}" * (isodd(k) && k < length(its) ? "\\hfill" : "\\par\\medskip"))
    end
    push!(L, "{\\raggedright\\noindent\\footnotesize Source: $(join(srcs, "; ")).\\par}", "\\end{figure}")
    return L
end

const QTITLE = Dict("6" => "Market simulation and welfare", "7" => "Demand moments and instruments", "8" => "Mixed logit by nested fixed point",
                    "9" => "Cost shifters and the supply side", "10" => "Merger simulation", "11" => "Newer instruments and substitution diagnostics")
qnum(q) = match(r"^\d+", q).match

function write_main()
    L = String[]
    push!(L, "\\documentclass[11pt]{article}", "\\usepackage[margin=1in]{geometry}", "\\usepackage{amsmath,amssymb}", "\\usepackage{graphicx,booktabs,adjustbox,float,subcaption}", "\\usepackage[T1]{fontenc}",
             "% Keep output/{figures,tables,overleaf}/ together in Overleaf and compile output/overleaf/main.tex", "\\graphicspath{{../figures/}}",
             "\\newcommand{\\figitem}[4]{\\begin{figure}[H]\\centering\\caption{#2}\\label{#3}\\includegraphics[width=0.8\\textwidth]{#1}\\par\\smallskip{\\raggedright\\noindent\\footnotesize Source: #4.\\par}\\end{figure}}",
             "\\newcommand{\\tabitem}[4]{\\begin{table}[H]\\centering\\caption{#2}\\label{#3}\\adjustbox{max width=\\textwidth}{\\input{../tables/#1}}\\par\\smallskip{\\raggedright\\noindent\\footnotesize Source: #4.\\par}\\end{table}}",
             "\\title{Economics 450-1, Problem Set 1: Figures and Tables}", "\\author{}", "\\date{}", "\\begin{document}", "\\maketitle", "")
    for qn in ("6", "7", "8", "9", "10", "11")
        push!(L, "\\section*{Question $qn: $(QTITLE[qn])}", "")
        parts = unique([it.q for it in ITEMS if qnum(it.q) == qn])
        for part in parts
            push!(L, "% ---- Q$part ----", "\\subsection*{$(part)}")
            for it in filter(i -> i.q == part, ITEMS)
                if it.kind == :fig && (gi = group_of(it.name)) !== nothing
                    it.name == GROUPS[gi][2][1] && push!(L, group_block(GROUPS[gi])...)
                    continue
                end
                label = (it.kind == :fig ? "fig:" : "tab:") * it.name
                cap = it.caption
                if it.kind == :fig
                    push!(L, "\\figitem{$(it.name).pdf}{$cap}{$label}{$(it.source)}")
                else
                    push!(L, "\\tabitem{$(it.name).tex}{$cap}{$label}{$(it.source)}")
                end
            end
            push!(L, "")
        end
    end
    push!(L, "\\end{document}")
    open("output/overleaf/main.tex", "w") do io
        foreach(l -> println(io, l), L)
    end
    return length(ITEMS)
end
write_main()

### Checks: every referenced file exists, labels are unique

In [ ]:
txt = read("output/overleaf/main.tex", String)
figs = filter(f -> !startswith(f, "#"), [m.captures[1] for m in eachmatch(r"\\includegraphics(?:\[[^\]]*\])?\{([^}]*)\}", txt)])
tabs = [m.captures[1] for m in eachmatch(r"\\tabitem\{([^}]*)\}", txt)]
labels = [m.captures[1] for m in eachmatch(r"\{((?:fig|tab):[^}]*)\}", txt)]
missing_files = vcat(["output/figures/$f" for f in figs if !isfile("output/figures/$f")], ["output/tables/$f" for f in tabs if !isfile("output/tables/$f")],
                     ["output/figures/$(replace(f, ".pdf" => ".png"))" for f in figs if !isfile("output/figures/$(replace(f, ".pdf" => ".png"))")])
@assert isempty(missing_files) "missing files: $missing_files"
@assert length(unique(labels)) == length(labels) "duplicate labels"
(figures = length(figs), tables = length(tabs))